<h3>Batch processing notebook based on 3D Cellpose dev logic (no napari visualization)</h3>

In [ ]:
from pathlib import Path
from tqdm import tqdm
import czifile
import numpy as np
import tifffile
import pandas as pd
from skimage.measure import regionprops_table
import plotly.express as px
import pyclesperanto_prototype as cle

from utils import (
    extract_scaling_metadata,
    predict_nuclei_labels,
    simulate_cytoplasm,
    _keep_objects_in_size_range,
)
from io_utils import (
    list_images,
    ensure_output_dir,
    load_precomputed_results_if_available,
)

cle.select_device("RTX")

### Configure experiment inputs
Set your input directory and analysis parameters.

In [ ]:
# Path where .czi images are stored
RAW_DATA_DIRECTORY = "./raw_data/"

# Channel index used for CellposeSAM-based 3D nuclei segmentation
NUCLEI_CHANNEL = 2

# YAP intensity channel used for nuclei/cytoplasm measurements
YAP_CHANNEL = 1

# Inclusive min/max nuclei volume used to filter predicted labels
# Set to None to skip label-size filtering
MIN_MAX_NUCLEI_VOLUME = (100, 1500)

# Cytoplasm ring dilation radius in voxels
CYTOPLASM_DILATION_RADIUS = 2

directory_path = Path(RAW_DATA_DIRECTORY)
input_folder_id = directory_path.stem
images = list_images(directory_path, "czi")
len(images)

In [ ]:
# Prepare output directories
nuclei_labels_dir = ensure_output_dir(RAW_DATA_DIRECTORY, input_folder_id, results_type="nuclei_labels")
bp_results_dir = ensure_output_dir(RAW_DATA_DIRECTORY, input_folder_id, results_type="bp_results")

dataframes = []

for image in tqdm(images):
    # Read path storing raw image and extract filename
    file_path = Path(image)
    filename = file_path.stem

    # Read the image file and remove singleton dimensions
    img = czifile.imread(image)
    img = img.squeeze()

    # Extract experiment, mouse, treatment and replica ids
    experiment_id = filename.split(" ")[0]
    mouse_id = filename.split(" ")[1]
    treatment_id = filename.split(" ")[2]
    replica_id = filename.split(" ")[-1]

    # Use YAP channel for intensity-based measurements
    yap_stack = img[YAP_CHANNEL, :, :, :]

    # Calculate anisotropy correction factor for Cellpose 3D inference
    scaling_x_um, scaling_y_um, scaling_z_um = extract_scaling_metadata(file_path)
    rescale_factor = scaling_z_um / np.mean([scaling_x_um, scaling_y_um])

    # Load existing labels, otherwise predict and save them
    nuclei_labels = load_precomputed_results_if_available(
        nuclei_labels_dir, filename, results_type="nuclei_labels"
    )

    if nuclei_labels is not None:
        if MIN_MAX_NUCLEI_VOLUME is not None:
            nuclei_labels = _keep_objects_in_size_range(nuclei_labels, MIN_MAX_NUCLEI_VOLUME)
    else:
        nuclei_labels = predict_nuclei_labels(
            img,
            rescale_factor,
            NUCLEI_CHANNEL,
            MIN_MAX_NUCLEI_VOLUME,
            visualize=False,
        )
        nuclei_labels_path = nuclei_labels_dir / f"{filename}_nuclei_labels.tif"
        tifffile.imwrite(nuclei_labels_path, nuclei_labels)

    # Simulate cytoplasm labels around each nucleus
    cytoplasm = simulate_cytoplasm(nuclei_labels, dilation_radius=CYTOPLASM_DILATION_RADIUS)

    # Extract regionprops from nuclei and simulated-cytoplasm labels
    props_nuclei = regionprops_table(
        label_image=nuclei_labels,
        intensity_image=yap_stack,
        properties=["label", "intensity_mean", "area"],
    )
    props_cytoplasm = regionprops_table(
        label_image=cytoplasm,
        intensity_image=yap_stack,
        properties=["label", "intensity_mean", "area"],
    )

    # Build and merge per-label dataframes
    df_nuclei = pd.DataFrame(props_nuclei)
    df_cytoplasm = pd.DataFrame(props_cytoplasm)

    df_nuclei.rename(columns={"intensity_mean": "intensity_mean_nuclei", "area": "area_nuclei"}, inplace=True)
    df_cytoplasm.rename(columns={"intensity_mean": "intensity_mean_cyto", "area": "area_cyto"}, inplace=True)

    merged_df = pd.merge(df_nuclei, df_cytoplasm, on="label")

    # Calculate nuclei/cytoplasm ratio of mean YAP intensity
    merged_df["nuclei_cyto_ratio"] = merged_df["intensity_mean_nuclei"] / merged_df["intensity_mean_cyto"]

    # Add image metadata ids
    merged_df.insert(0, "experiment_id", experiment_id)
    merged_df.insert(1, "mouse_id", mouse_id)
    merged_df.insert(2, "treatment_id", treatment_id)
    merged_df.insert(3, "replica_id", replica_id)

    dataframes.append(merged_df)

In [ ]:
# Concatenate per-image dataframes
final_df = pd.concat(dataframes, ignore_index=True)

# Save per-label results
per_label_path = bp_results_dir / f"per_label_results_{input_folder_id}_MIN{MIN_MAX_NUCLEI_VOLUME[0]}_MAX{MIN_MAX_NUCLEI_VOLUME[1]}_DIL{CYTOPLASM_DILATION_RADIUS}.csv"
final_df.to_csv(per_label_path, index=False)
per_label_path

In [ ]:
# Group by organoid identifiers and compute per-organoid averages
grouped_df = final_df.groupby(["experiment_id", "mouse_id", "treatment_id", "replica_id"]).agg({
    "intensity_mean_nuclei": "mean",
    "area_nuclei": "mean",
    "intensity_mean_cyto": "mean",
    "area_cyto": "mean",
    "nuclei_cyto_ratio": "mean",
}).reset_index()

average_path = bp_results_dir / f"average_nuclear_cyto_intensity_{input_folder_id}_MIN{MIN_MAX_NUCLEI_VOLUME[0]}_MAX{MIN_MAX_NUCLEI_VOLUME[1]}_DIL{CYTOPLASM_DILATION_RADIUS}.csv"
grouped_df.to_csv(average_path, index=False)
average_path

In [ ]:
grouped_df.head()

In [ ]:
# Create the box plot
fig = px.box(
    grouped_df,
    x="treatment_id",
    y="intensity_mean_nuclei",
    color="treatment_id",
    points="all",
    hover_data=["experiment_id", "mouse_id", "replica_id"],
    title="YAP Nuclear Intensity Average by Treatment ID",
)

fig.show()

In [ ]:
# Create the box plot
fig = px.box(
    grouped_df,
    x="treatment_id",
    y="intensity_mean_cyto",
    color="treatment_id",
    points="all",
    hover_data=["experiment_id", "mouse_id", "replica_id"],
    title="YAP Cytoplasmic Intensity Average by Treatment ID",
)

fig.show()

In [ ]:
# Create the box plot
fig = px.box(
    grouped_df,
    x="treatment_id",
    y="nuclei_cyto_ratio",
    color="treatment_id",
    points="all",
    hover_data=["experiment_id", "mouse_id", "replica_id"],
    title="YAP Nuclear/Cytoplasmic signal Ratio by Treatment ID",
)

fig.show()